# 🚀 Training MG-MMKT v2 on Kaggle (2x NVIDIA Tesla T4 GPUs)
**Mô hình:** Second-Generation Masked-Gated Multimodal Knowledge Tracing (MG-MMKT v2)  
**Tập dữ liệu:** Preprocessed XES3G5M (Chinese Mathematics Multimodal Knowledge Tracing — 5,549,184 tương tác)  
**Mục tiêu:** Tận dụng tối đa cụm 2x GPU NVIDIA Tesla T4 (mỗi GPU 16GB VRAM, tổng cộng 32GB VRAM) để huấn luyện tối ưu tốc độ, hỗ trợ chạy song song toàn diện **5-Fold Cross-Validation cho Full Model và TẤT CẢ 5 khâu thực nghiệm loại suy (Ablation Studies)**, đối sánh trực tiếp với đường cơ sở MG-MMKT v1 (Mean: 0.8290, Ensemble: 0.8321), đồng thời trích xuất báo cáo **Khả năng Giải thích 3 Cấp độ (Multi-Level Explainability)**.

---

### ⚙️ Hướng Dẫn Cấu Hình Kaggle Notebook (Bắt Buộc):
1. **Accelerator:** Ở thanh menu bên phải (**Session options**), chọn **GPU T4 x 2**.
2. **Internet:** Chuyển sang **Internet ON** (bắt buộc để tải packages và clone/pull git repository).
3. **Data Input:** Nhấn **+ Add Input**, chọn Dataset XES3G5M đã tiền xử lý của bạn (được gắn vào thư mục `/kaggle/input/...`).

In [ ]:
# [1] Kiểm tra cấu hình phần cứng & Cấu hình chống phân mảnh VRAM
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi

import torch
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
gpu_count = torch.cuda.device_count()
print(f"Số lượng GPU nhận diện: {gpu_count}")

for i in range(gpu_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  [GPU {i}] {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB | Compute Capability: {props.major}.{props.minor}")

assert gpu_count >= 1, "⚠️ Vui lòng chuyển Accelerator sang GPU T4 x 2 trong cài đặt Kaggle!"

In [ ]:
# [2] Thiết lập môi trường làm việc & Clone/Pull repository từ GitHub
import os
import sys

REPO_URL = "https://github.com/nbao0208/kt-mmkt-research.git"
REPO_DIR = "/kaggle/working/kt-mmkt-research"

if os.path.exists("src/models/mmkt/mg_mmkt_v2.py"):
    print("✅ Bạn đang đứng trực tiếp trong thư mục repository kt-mmkt-research.")
    !git pull origin main
elif os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"✅ Đã chuyển thư mục làm việc đến: {REPO_DIR}")
    !git pull origin main
else:
    print(f"📥 Đang clone repository từ {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    print(f"✅ Đã clone thành công và chuyển vào: {os.getcwd()}")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Working Directory:", os.getcwd())

In [ ]:
# [3] Cài đặt & cập nhật các thư viện phụ thuộc (Dependencies)
!pip install -q -r requirements.txt
!pip install -q seaborn matplotlib scikit-learn tabulate pandas
print("✅ Hoàn tất cài đặt toàn bộ gói phụ thuộc!")

In [ ]:
# [4] Tự động phát hiện và liên kết (Symlink) tập dữ liệu XES3G5M từ /kaggle/input/
import glob
import torch
from pathlib import Path

# 1. Tìm thư mục chứa file static item bank hoặc sequence files
candidates = glob.glob("/kaggle/input/**/sequences_fold_0.pt", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/input/**/static_item_bank.pt", recursive=True)
if not candidates:
    candidates = glob.glob("/kaggle/input/**/mg_static_item_bank.pt", recursive=True)

if not candidates:
    raise FileNotFoundError(
        "❌ Không tìm thấy tập dữ liệu XES3G5M trong /kaggle/input/!\n"
        "👉 Vui lòng nhấn nút '+ Add Input' trên giao diện Kaggle và thêm Dataset XES3G5M đã tiền xử lý của bạn."
    )

input_data_dir = Path(candidates[0]).parent
print(f"🔍 Đã tìm thấy thư mục chứa dữ liệu tại: {input_data_dir}")

target_dir = Path("data/processed/XES3G5M")
target_dir.mkdir(parents=True, exist_ok=True)

# 2. Tạo symlink cho toàn bộ file .pt và .json
linked_count = 0
for ext in ["*.pt", "*.json"]:
    for src_file in input_data_dir.glob(ext):
        dst_file = target_dir / src_file.name
        if dst_file.exists() or dst_file.is_symlink():
            dst_file.unlink()
        dst_file.symlink_to(src_file)
        linked_count += 1

print(f"✅ Đã tạo symbolic link cho {linked_count} tệp dữ liệu vào {target_dir}.")

# 3. Tối ưu hóa: Nếu có static_item_bank.pt (4.6GB) nhưng chưa có mg_static_item_bank.pt (~48MB),
# tự động trích xuất để nạp siêu nhanh và tiết kiệm RAM tối đa cho MG-MMKT v2!
from src.data.xes3g5m_dataset import create_mg_static_item_bank

full_bank_path = target_dir / "static_item_bank.pt"
mg_bank_path = target_dir / "mg_static_item_bank.pt"

if not mg_bank_path.exists() and full_bank_path.exists():
    print("⚡ Đang trích xuất mg_static_item_bank.pt nhẹ (~48 MB) từ file 4.6 GB...")
    create_mg_static_item_bank(str(full_bank_path), str(mg_bank_path))
    print("✅ Trích xuất hoàn tất!")

print("\n📁 Danh sách các tệp dữ liệu sẵn sàng cho huấn luyện:")
!ls -lh data/processed/XES3G5M

In [ ]:
# [5] Chạy Sanity Check Unit Tests để đảm bảo kiến trúc MG-MMKT v2 và giao thức pyKT hoạt động hoàn hảo
!python -m pytest tests/test_mg_mmkt_v2_model.py tests/test_pykt_protocol.py -v
print("🎉 Toàn bộ Unit Tests cho kiến trúc MG-MMKT v2 đã vượt qua thành công!")

## 🧠 Kiến Trúc MG-MMKT v2 & Ma Trận 6 Khâu Thực Nghiệm Loại Suy (Ablation Matrix)

Theo tài liệu đặc tả kiến trúc `docs/models/MG-MMKTv2.md`, **MG-MMKT v2** khắc phục triệt để các hạn chế của v1 thông qua 4 tầng kiến trúc và cơ chế Học với Dữ liệu Đặc quyền (LUPI):
1. **Tầng 1 — Tách Biệt Cấu Trúc & Khôi Phục Có Điều Kiện Kỹ Năng:** Cổng tần suất $g_{\text{freq}}$ kết hợp Question ID Dropout ngẫu nhiên ở cấp bước triệt tiêu hiện tượng ghi nhớ câu hỏi phổ biến; cơ chế bù khuyết vector $\tilde{v}_m$ học được từ kỹ năng thành phần $e_{\mathcal{C}}$ nằm bên trong `LayerNorm`.
2. **Tầng 2 — Cổng Kép Nhận Thức Khuyết Thiếu:** Kết hợp Cổng Softmax Thích ứng (với tham số phạt học được $\delta \in [10^{-4}, 10.0]$) và Cổng GLU tinh chỉnh kênh, đi kèm khối FFN phi tuyến chuyển tiếp.
3. **Tầng 3 — Hợp Nhất Lịch Sử Nhận Thức & Ước Lượng Nhu Cầu Ẩn (LUPI):** Tách riêng hai nhánh chuyển đổi nhận thức $\Delta y^{(1)}$ (Mastery) và $\Delta y^{(0)}$ (Remedial). Mạng học sinh nhân quả $\hat{z}_t^s = f_{\text{cog}}(x_t)$ ước lượng tải nhận thức thuật toán của lời giải và được chưng cất từ biểu diễn Teacher sạch $z_t^{s, \text{clean}}$.
4. **Tầng 4 — Chú Ý Nhân Quả Tách Biệt Có Suy Giảm Tương Đối:** Ma trận chú ý $(L-1) \times (L-1)$ triệt tiêu nguy cơ NaN tại $t=1$, suy giảm đa thang đo monotonic $-\theta_h \ln(1 + (t-\tau))$, giữ trạng thái kiến thức $h_t$ thuần khiết và đầu đọc đa tương tác $5d$.

### 🔬 Bảng Ma Trận Loại Suy (Ablation Matrix):
| Cấu hình | Tên Cấu Hình | Cờ CLI | Mô Tả Kỹ Thuật | Mục Tiêu Khoa Học |
| :--- | :--- | :--- | :--- | :--- |
| **Config 1** | **Full MG-MMKT v2** | `--ablation 1` | Đầy đủ 4 tầng + Chưng cất đặc quyền $\mathcal{L}_{\text{priv}}$ + Phạt khuyết thiếu $\mathcal{L}_{\text{gate}}$ | Mô hình hoàn chỉnh đề xuất tối ưu hóa toàn diện |
| **Config 2** | **w/o LUPI Distillation** | `--ablation 2` (`--wo-lupi`) | Tắt chưng cất đặc quyền ($\lambda_{\text{priv}} = 0$) và bỏ vector $\hat{z}_t^s$ trong readout | Đo lường giá trị của cơ chế ước lượng tải nhận thức thuật toán |
| **Config 3** | **w/o Dual-Pathway History** | `--ablation 3` (`--wo-dual-pathway`) | Dùng chung một hàm chuyển đổi lịch sử cho cả $r_\tau=1$ và $r_\tau=0$ | Đo lường hiệu quả của việc phân tách nhánh Mastery và Remedial |
| **Config 4** | **w/o GLU & Residual FFN** | `--ablation 4` (`--wo-glu-ffn`) | Tắt cổng GLU và FFN phi tuyến ở Tầng 2, chỉ giữ Softmax Gate | Kiểm chứng vai trò của biểu diễn phi tuyến tinh chỉnh kênh |
| **Config 5** | **w/o Disentangled QK & Relative Forgetting** | `--ablation 5` | Chiếu Key từ lịch sử $y$, tắt suy giảm $-\theta_h \ln(1+(t-\tau))$ và dùng nhúng vị trí tuyệt đối | Chứng minh ưu thế của chú ý tách biệt câu hỏi và suy giảm theo thời gian |
| **Config 6** | **w/o Structural Decoupling & KC Imputation** | `--ablation 6` | Tắt cổng tần suất & ID Dropout, thay khôi phục KC bằng Zero-imputation | Kiểm chứng khả năng chống quá khớp câu hỏi và xử lý khuyết thiếu |

### ⚡ Chiến Lược Tận Dụng 2x GPU NVIDIA Tesla T4:
- **Phân bổ 5-Fold Song Song:** Khi bật `--dual-gpu`, luồng chính khởi tạo bất đồng bộ: **GPU 0** xử lý Folds [0, 1, 2] và **GPU 1** xử lý Folds [3, 4]. Cả hai GPU chạy độc lập 100% công suất, rút ngắn thời gian chạy 5-fold xuống còn ~10-12 phút mỗi mô hình!
- Sau khi 2 tiến trình con hoàn thành, luồng chính tự động kích hoạt chế độ đánh giá (`--eval-only`) để thu thập xác suất dự đoán trên $1.099.197$ tương tác kiểm thử và tổng hợp báo cáo 5-Fold Soft Ensemble.

In [ ]:
# ==============================================================================
# [Cấu Hình Huấn Luyện Toàn Cục] Thiết lập chế độ chạy 5-Fold cho Full Model & Ablations
# ==============================================================================
# 👉 Đặt RUN_5FOLD_FULL = True: Huấn luyện đầy đủ 5-Fold cho Full MG-MMKT v2 trên 2x GPU T4
# 👉 Đặt RUN_ALL_ABLATIONS = True: Huấn luyện tiếp toàn bộ 5 khâu Ablation
#    (Thời gian: ~10-12 phút cho Full Model; ~50-60 phút cho toàn bộ ma trận Ablation)
RUN_5FOLD_FULL = True
RUN_ALL_ABLATIONS = True

NUM_EPOCHS = 35
BATCH_SIZE = 64
LR = 1e-3
LAMBDA_PRIV = 0.10
LAMBDA_GATE = 0.05

print(f"⚙️ Cấu hình huấn luyện:")
print(f"   Full Model 5-Fold: {'BẬT (2x T4 Parallel)' if RUN_5FOLD_FULL else 'TẮT'}")
print(f"   Ablations 5-Fold: {'BẬT (5 cấu hình)' if RUN_ALL_ABLATIONS else 'TẮT'}")
print(f"   Epochs: {NUM_EPOCHS} | Batch Size: {BATCH_SIZE} | LR: {LR}")
print(f"   Loss Weights: lambda_priv={LAMBDA_PRIV} | lambda_gate={LAMBDA_GATE}")

In [ ]:
# ==============================================================================
# [Kịch bản 1] Huấn luyện Mô hình Đề Xuất Full MG-MMKT v2 (Config 1)
# ==============================================================================
# - Chạy 5-Fold Cross-Validation song song trên 2x GPU T4 (GPU 0: Folds 0,1,2 | GPU 1: Folds 3,4)
# - Tự động tổng hợp bảng kết quả 5-Fold (Mean ± Std) và 5-Fold Soft Ensemble chuẩn pyKT
# - Đối sánh trực tiếp mức độ tăng trưởng (Delta AUC) so với MG-MMKT v1 (Mean 0.8290, Ensemble 0.8321)

import os

os.makedirs("outputs/checkpoints/v2_full_5fold", exist_ok=True)
os.makedirs("outputs/reports/v2_full_5fold", exist_ok=True)

if RUN_5FOLD_FULL:
    print("🚀 Đang khởi động huấn luyện 5-Fold song song cho Full MG-MMKT v2 trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt_v2.py \
        --dual-gpu \
        --ablation 1 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/v2_full_5fold \
        --output-report-dir outputs/reports/v2_full_5fold
    print("✅ Hoàn thành huấn luyện và tổng hợp 5-Fold cho Full MG-MMKT v2!")

## 🧪 Thực Hiện Toàn Bộ 5 Khâu Thực Nghiệm Loại Suy (Ablation Matrix)

Các ô lệnh dưới đây tự động chạy 5-Fold Cross-Validation trên 2x GPU T4 cho từng cấu hình loại suy:
- **Đợt 1:** Config 2 (`w/o LUPI Distillation`) và Config 3 (`w/o Dual-Pathway History`)
- **Đợt 2:** Config 4 (`w/o GLU & Residual FFN`) và Config 5 (`w/o Disentangled QK & Relative Forgetting`)
- **Đợt 3:** Config 6 (`w/o Structural Decoupling & KC Imputation`)

In [ ]:
# ==============================================================================
# [Ablation Đợt 1] Config 2 (w/o LUPI) & Config 3 (w/o Dual-Pathway History)
# ==============================================================================
if RUN_ALL_ABLATIONS:
    print("🚀 [1/2] Đang huấn luyện 5-Fold cho Config 2 (w/o LUPI) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt_v2.py \
        --dual-gpu \
        --ablation 2 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation2_wolupi_5fold \
        --output-report-dir outputs/reports/ablation2_wolupi_5fold

    print("\n🚀 [2/2] Đang huấn luyện 5-Fold cho Config 3 (w/o Dual-Pathway History) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt_v2.py \
        --dual-gpu \
        --ablation 3 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation3_wodualpath_5fold \
        --output-report-dir outputs/reports/ablation3_wodualpath_5fold
    print("✅ Hoàn thành Ablation Đợt 1!")

In [ ]:
# ==============================================================================
# [Ablation Đợt 2] Config 4 (w/o GLU-FFN) & Config 5 (w/o Disentangled QK & Relative Forgetting)
# ==============================================================================
if RUN_ALL_ABLATIONS:
    print("🚀 [1/2] Đang huấn luyện 5-Fold cho Config 4 (w/o GLU & Residual FFN) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt_v2.py \
        --dual-gpu \
        --ablation 4 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation4_woglu_5fold \
        --output-report-dir outputs/reports/ablation4_woglu_5fold

    print("\n🚀 [2/2] Đang huấn luyện 5-Fold cho Config 5 (w/o Disentangled QK & Relative Forgetting) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt_v2.py \
        --dual-gpu \
        --ablation 5 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation5_woforgetting_5fold \
        --output-report-dir outputs/reports/ablation5_woforgetting_5fold
    print("✅ Hoàn thành Ablation Đợt 2!")

In [ ]:
# ==============================================================================
# [Ablation Đợt 3] Config 6 (w/o Structural Decoupling & KC Imputation)
# ==============================================================================
if RUN_ALL_ABLATIONS:
    print("🚀 Đang huấn luyện 5-Fold cho Config 6 (w/o Structural Decoupling & KC Imputation) trên 2x GPU T4...")
    !python -u scripts/run_5fold_mg_mmkt_v2.py \
        --dual-gpu \
        --ablation 6 \
        --num-epochs {NUM_EPOCHS} \
        --batch-size {BATCH_SIZE} \
        --lr {LR} \
        --save-base-dir outputs/checkpoints/ablation6_wostruct_5fold \
        --output-report-dir outputs/reports/ablation6_wostruct_5fold
    print("✅ Hoàn thành toàn bộ ma trận Ablation Study 5-Fold!")

In [ ]:
# [11] Đánh giá chi tiết tập Test (1,099,197 tương tác) & Báo cáo Khả năng Giải thích 3 Cấp độ
import os

os.makedirs("outputs/reports", exist_ok=True)

ckpt_full = "outputs/checkpoints/v2_full_5fold/mg_mmkt_v2_fold0/best_model.pt"
if not os.path.exists(ckpt_full):
    # Fallback to single fold path if trained individually
    ckpt_full = "outputs/checkpoints/mg_mmkt_v2_fold0/best_model.pt"

if os.path.exists(ckpt_full):
    print("🔍 Đang chạy đánh giá phân tầng và trích xuất báo cáo giải thích 3 cấp độ cho Full MG-MMKT v2...")
    !python -u scripts/evaluate_mg_mmkt_v2.py \
        --checkpoint {ckpt_full} \
        --val-fold 0 \
        --device cuda:0 \
        --output-report outputs/reports/eval_mg_mmkt_v2_full.json
    print("✅ Báo cáo đánh giá chi tiết đã hoàn thành và lưu tại outputs/reports/eval_mg_mmkt_v2_full.json!")
else:
    print(f"⚠️ Chưa tìm thấy checkpoint: {ckpt_full}. Vui lòng chạy kịch bản huấn luyện trước.")

In [ ]:
# [12] Bảng Tổng Hợp So Sánh Toàn Bộ Các Mô Hình & Đối Sánh Trực Tiếp Với MG-MMKT v1
import json
import os
import pandas as pd
from IPython.display import display

# Đường cơ sở chuẩn của MG-MMKT v1 đã kiểm toán trên XES3G5M
V1_BENCHMARK = {
    "mean_test_auc": 0.8290,
    "std_test_auc": 0.0001,
    "ensemble_test_auc": 0.8321,
}

configs_5fold = [
    ("Full MG-MMKT v2 (Đề xuất)", "outputs/reports/v2_full_5fold/5fold_summary_abl1.json", "outputs/reports/v2_full_5fold/5fold_summary.json", "Mô hình hoàn chỉnh đề xuất"),
    ("Ablation 2 (w/o LUPI)", "outputs/reports/ablation2_wolupi_5fold/5fold_summary_abl2.json", "outputs/reports/ablation2_wolupi_5fold/5fold_summary.json", "Tắt chưng cất đặc quyền & tải nhận thức"),
    ("Ablation 3 (w/o Dual-Pathway)", "outputs/reports/ablation3_wodualpath_5fold/5fold_summary_abl3.json", "outputs/reports/ablation3_wodualpath_5fold/5fold_summary.json", "Dùng chung hàm chuyển đổi lịch sử"),
    ("Ablation 4 (w/o GLU & FFN)", "outputs/reports/ablation4_woglu_5fold/5fold_summary_abl4.json", "outputs/reports/ablation4_woglu_5fold/5fold_summary.json", "Bỏ GLU & FFN phi tuyến sau cổng"),
    ("Ablation 5 (w/o Relative Forgetting)", "outputs/reports/ablation5_woforgetting_5fold/5fold_summary_abl5.json", "outputs/reports/ablation5_woforgetting_5fold/5fold_summary.json", "SAKT query/key + nhúng vị trí tuyệt đối"),
    ("Ablation 6 (w/o Struct & KC-Imput)", "outputs/reports/ablation6_wostruct_5fold/5fold_summary_abl6.json", "outputs/reports/ablation6_wostruct_5fold/5fold_summary.json", "Tắt ID dropout + zero-imputation"),
]

results = []
full_mean_auc = None

for name, p1, p2, desc in configs_5fold:
    path = p1 if (p1 and os.path.exists(p1)) else (p2 if (p2 and os.path.exists(p2)) else None)
    if path:
        with open(path) as f:
            d = json.load(f)
        mean_auc = d.get("mean_test_auc", 0.0)
        std_auc = d.get("std_test_auc", 0.0)
        mean_acc = d.get("mean_test_acc", 0.0)
        std_acc = d.get("std_test_acc", 0.0)
        ens_auc = d.get("ensemble_metrics", {}).get("auc", 0.0)
        
        if full_mean_auc is None and "Full" in name:
            full_mean_auc = mean_auc
            
        delta_v1 = f"{mean_auc - V1_BENCHMARK['mean_test_auc']:+.4f}"
        delta_full = f"{mean_auc - full_mean_auc:+.4f}" if full_mean_auc else "--"
        
        results.append({
            "Model Variant": name,
            "5-Fold Single AUC": f"{mean_auc:.4f} ± {std_auc:.4f}",
            "5-Fold Single ACC": f"{mean_acc:.4f} ± {std_acc:.4f}",
            "5-Fold Ensemble AUC": f"{ens_auc:.4f}" if ens_auc > 0 else "--",
            "Δ AUC (vs v1)": delta_v1,
            "Δ AUC (vs Full v2)": delta_full,
            "Description": desc
        })

if results:
    df_results = pd.DataFrame(results)
    print("\n🏆 BẢNG TỔNG HỢP SO SÁNH 5-FOLD CROSS-VALIDATION MG-MMKT v2 TRÊN XES3G5M:")
    display(df_results)
    df_results.to_markdown("outputs/reports/mg_mmkt_v2_ablation_synthesis.md", index=False)
    df_results.to_json("outputs/reports/mg_mmkt_v2_ablation_synthesis.json", indent=2, orient="records")
    print("\n✅ Báo cáo so sánh đã được lưu tại outputs/reports/mg_mmkt_v2_ablation_synthesis.md")
else:
    print("ℹ️ Chưa có file tóm tắt 5-Fold nào được tìm thấy.")

In [ ]:
# [13] Trực quan hóa Biểu Đồ Nghiên Cứu Chất Lượng Cao (Publication-Ready Figures)
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

os.makedirs("outputs/reports/figures", exist_ok=True)
sns.set_theme(style="whitegrid", font_scale=1.05)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
ax1, ax2, ax3, ax4 = axes.flatten()

# -------------------------------------------------------------------------
# (a) Tiến trình Validation AUC theo Epochs (Fold 0)
# -------------------------------------------------------------------------
histories = [
    ("Full MG-MMKT v2", "outputs/checkpoints/v2_full_5fold/mg_mmkt_v2_fold0/training_summary.json", "#1f77b4"),
    ("w/o LUPI", "outputs/checkpoints/ablation2_wolupi_5fold/mg_mmkt_v2_fold0/training_summary.json", "#ff7f0e"),
    ("w/o Dual-Pathway", "outputs/checkpoints/ablation3_wodualpath_5fold/mg_mmkt_v2_fold0/training_summary.json", "#2ca02c"),
    ("w/o GLU-FFN", "outputs/checkpoints/ablation4_woglu_5fold/mg_mmkt_v2_fold0/training_summary.json", "#d62728"),
    ("w/o Relative Forgetting", "outputs/checkpoints/ablation5_woforgetting_5fold/mg_mmkt_v2_fold0/training_summary.json", "#9467bd"),
    ("w/o Struct & KC", "outputs/checkpoints/ablation6_wostruct_5fold/mg_mmkt_v2_fold0/training_summary.json", "#8c564b"),
]

plotted_a = False
for label, path, color in histories:
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        h_list = data.get("history", [])
        epochs = [h.get("epoch", i+1) for i, h in enumerate(h_list)]
        val_aucs = [h.get("val_auc", 0) for h in h_list]
        ax1.plot(epochs, val_aucs, label=label, color=color, linewidth=2.0, marker="o", markersize=3.0)
        plotted_a = True

if plotted_a:
    ax1.set_title("(a) Validation ROC-AUC Progression across Epochs", fontsize=12, fontweight="bold", pad=10)
    ax1.set_xlabel("Epoch", fontsize=11)
    ax1.set_ylabel("Validation ROC-AUC", fontsize=11)
    ax1.legend(loc="lower right", frameon=True, fontsize=9)
    ax1.grid(True, linestyle="--", alpha=0.6)

# -------------------------------------------------------------------------
# (b) So Sánh 5-Fold Test ROC-AUC (Mean ± Std) với MG-MMKT v1
# -------------------------------------------------------------------------
report_sources = [
    ("MG-MMKT v1 (Baseline)", 0.8290, 0.0001, "#7f7f7f"),
    ("Full v2 (Proposed)", None, None, "#1f77b4"),
    ("w/o LUPI", None, None, "#ff7f0e"),
    ("w/o Dual-Pathway", None, None, "#2ca02c"),
    ("w/o GLU-FFN", None, None, "#d62728"),
    ("w/o Rel-Forgetting", None, None, "#9467bd"),
    ("w/o Struct & KC", None, None, "#8c564b"),
]

v2_reports = {
    "Full v2 (Proposed)": "outputs/reports/v2_full_5fold/5fold_summary.json",
    "w/o LUPI": "outputs/reports/ablation2_wolupi_5fold/5fold_summary.json",
    "w/o Dual-Pathway": "outputs/reports/ablation3_wodualpath_5fold/5fold_summary.json",
    "w/o GLU-FFN": "outputs/reports/ablation4_woglu_5fold/5fold_summary.json",
    "w/o Rel-Forgetting": "outputs/reports/ablation5_woforgetting_5fold/5fold_summary.json",
    "w/o Struct & KC": "outputs/reports/ablation6_wostruct_5fold/5fold_summary.json",
}

m_names, m_aucs, m_stds, m_cols = [], [], [], []
for name, base_auc, base_std, col in report_sources:
    if base_auc is not None:
        m_names.append(name)
        m_aucs.append(base_auc)
        m_stds.append(base_std)
        m_cols.append(col)
    elif name in v2_reports and os.path.exists(v2_reports[name]):
        with open(v2_reports[name]) as f:
            d = json.load(f)
        m_names.append(name)
        m_aucs.append(d.get("mean_test_auc", 0.0))
        m_stds.append(d.get("std_test_auc", 0.0))
        m_cols.append(col)

if m_aucs:
    bars = ax2.bar(m_names, m_aucs, yerr=m_stds, capsize=4, color=m_cols, width=0.55, edgecolor="black", linewidth=0.8)
    ax2.set_title("(b) 5-Fold Test ROC-AUC Comparison (Mean ± Std)", fontsize=12, fontweight="bold", pad=10)
    ax2.set_ylabel("Test ROC-AUC", fontsize=11)
    ax2.set_ylim(max(0.78, min(m_aucs) - 0.02), max(m_aucs) + 0.02)
    for bar, val in zip(bars, m_aucs):
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 0.002, f"{val:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=9.5)
    ax2.tick_params(axis="x", rotation=25)
    ax2.grid(axis="y", linestyle="--", alpha=0.6)

# -------------------------------------------------------------------------
# (c) Đóng Góp Phương Thức Tầng 2 (Alpha Attributions)
# -------------------------------------------------------------------------
eval_full_path = "outputs/reports/eval_mg_mmkt_v2_full.json"
if os.path.exists(eval_full_path):
    with open(eval_full_path) as f:
        ev_data = json.load(f)
    l1 = ev_data.get("level1_modality_attribution", {}).get("question_level", {})
    w_img = l1.get("with_image", {})
    t_only = l1.get("text_only", {})
    
    cats = ["Questions With Image", "Text-Only Questions"]
    a_id = [w_img.get("mean_alpha_id", 0.35), t_only.get("mean_alpha_id", 0.45)]
    a_txt = [w_img.get("mean_alpha_txt", 0.35), t_only.get("mean_alpha_txt", 0.55)]
    a_img = [w_img.get("mean_alpha_img", 0.30), t_only.get("mean_alpha_img", 0.00)]
    
    x = np.arange(len(cats))
    width = 0.25
    ax3.bar(x - width, a_id, width, label="Alpha ID", color="#4e79a7", edgecolor="black", linewidth=0.8)
    ax3.bar(x, a_txt, width, label="Alpha Text", color="#59a14f", edgecolor="black", linewidth=0.8)
    ax3.bar(x + width, a_img, width, label="Alpha Image", color="#e15759", edgecolor="black", linewidth=0.8)
    
    ax3.set_title("(c) Level 1 Gated Modality Attribution (Alpha Weights)", fontsize=12, fontweight="bold", pad=10)
    ax3.set_xticks(x)
    ax3.set_xticklabels(cats, fontsize=10.5)
    ax3.set_ylabel("Attribution Weight", fontsize=11)
    ax3.set_ylim(0.0, 1.0)
    ax3.legend(loc="upper right", frameon=True, fontsize=9.5)
    ax3.grid(axis="y", linestyle="--", alpha=0.6)

# -------------------------------------------------------------------------
# (d) Thống Kê Toàn Cục Tải Nhận Thức Bậc 3 (Level 3 Cognitive Alignment)
# -------------------------------------------------------------------------
if os.path.exists(eval_full_path):
    l3_stats = ev_data.get("level3_global_statistics", {})
    prof = l3_stats.get("predicted_proficient", {})
    strug = l3_stats.get("predicted_struggling", {})
    
    groups = ["Proficient (p >= 0.5)", "Struggling (p < 0.5)"]
    c_cos = [prof.get("mean_concept_cosine", 0.55), strug.get("mean_concept_cosine", 0.18)]
    d_cos = [prof.get("mean_cognitive_demand_cosine", 0.52), strug.get("mean_cognitive_demand_cosine", 0.15)]
    
    x = np.arange(len(groups))
    width = 0.35
    ax4.bar(x - width/2, c_cos, width, label="Concept Cosine cos(h, x)", color="#2b5c8f", edgecolor="black", linewidth=0.8)
    ax4.bar(x + width/2, d_cos, width, label="Cognitive Demand Cosine cos(h, z_hat)", color="#d95f02", edgecolor="black", linewidth=0.8)
    
    ax4.set_title("(d) Level 3 Cognitive State Diagnosis across Test Set", fontsize=12, fontweight="bold", pad=10)
    ax4.set_xticks(x)
    ax4.set_xticklabels(groups, fontsize=10.5)
    ax4.set_ylabel("Cosine Similarity", fontsize=11)
    ax4.set_ylim(0.0, 1.0)
    ax4.legend(loc="upper right", frameon=True, fontsize=9.5)
    ax4.grid(axis="y", linestyle="--", alpha=0.6)

plt.tight_layout()
fig_path = "outputs/reports/figures/mg_mmkt_v2_comprehensive_analysis.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Biểu đồ nghiên cứu độ phân giải cao đã được lưu tại: {fig_path}")

In [ ]:
# [14] Đóng gói toàn bộ Checkpoints, Reports, Logs và Figures thành file ZIP để tải về
import os

!zip -q -r /kaggle/working/mg_mmkt_v2_results.zip outputs/checkpoints outputs/reports outputs/logs
zip_size_mb = os.path.getsize("/kaggle/working/mg_mmkt_v2_results.zip") / (1024**2)
print(f"🎉 Đã nén thành công toàn bộ kết quả vào: /kaggle/working/mg_mmkt_v2_results.zip ({zip_size_mb:.2f} MB)")
print("👉 Bạn có thể tải file này trực tiếp từ khung 'Output' bên phải màn hình giao diện Kaggle!")